## Tables & figures

In [13]:
"""
gen_figures.py  —  Visualization figures for AGOPNullSpace paper
Run: python gen_figures.py
Outputs (all PNG):
  fig1_method_overview.png   — Pipeline diagram
  fig2_dsr_sweep.png         — DSR vs steering strength sweep
  fig3_cipher_highlight.png  — Cipher attack bar comparison
  fig4_radar.png             — Radar: safety vs utility
  fig5_agop_concept.png      — AGOP direction vs DiffMean concept art
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter

# ─── Palette ──────────────────────────────────────────────────────────────────
BG      = "#FFFFFF"      # Nền trắng
PANEL   = "#F8F9FA"      # Panel xám nhạt
BORDER  = "#DEE2E6"      # Viền xám
PRI     = "#1A1A2E"      # Chữ chính - đậm
SEC     = "#4A5568"      # Chữ phụ - xám đậm
BLUE    = "#1E6F9F"      # Xanh đậm
GREEN   = "#2E8B57"      # Xanh lá đậm
ORANGE  = "#E67E22"      # Cam đậm
RED     = "#C0392B"      # Đỏ đậm
PURPLE  = "#8E44AD"      # Tím đậm
TEAL    = "#008080"      # Xanh ngọc
GOLD    = "#D4AF37"      # Vàng
FONT    = "DejaVu Sans"
MONO    = "DejaVu Sans Mono"


def save(name, dpi=180):
    plt.savefig(f"/home/workspace/mad_workspace/llm/AlphaSteer/figures/{name}", dpi=dpi,
                bbox_inches="tight", facecolor=BG)
    print(f"✓ {name} saved")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# FIG 1 — Method pipeline overview
# ══════════════════════════════════════════════════════════════════════════════

def fig1_pipeline():
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.axis("off")
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 5)

    title = "AGOPNullSpace — Method Pipeline"
    ax.text(7, 4.65, title, ha="center", va="center", fontsize=13,
            color=PRI, fontweight="bold", fontfamily=FONT)

    # ── Step boxes ────────────────────────────────────────────────────────────
    steps = [
        (1.1, "① Collect\nActivations",
         "Harmful / benign\nprompts → H_m, H_b",
         BLUE),
        (3.6, "② Run RFM\n(AGOP loop)",
         "KRR + AGOP metric\nT iterations → M_T",
         PURPLE),
        (6.1, "③ Extract\nr_rfm",
         "top_eigenvec(M_T)\n→ refusal direction",
         TEAL),
        (8.6, "④ Null-space\nProjection",
         "P̂ = Û Ûᵀ (60% low\neigenvecs of Cov_b)",
         ORANGE),
        (11.1, "⑤ Steering\nMatrix Δ*",
         "AlphaSteer Eq. 9\nΔ* = R H_mᵀ P̂ᵀ (…)⁺",
         GREEN),
    ]

    box_w, box_h = 2.1, 2.0
    y_box = 1.4
    for x, title_s, body, color in steps:
        # glow
        for alpha, pad in [(0.08, 0.18), (0.15, 0.10), (0.25, 0.04)]:
            rect = FancyBboxPatch((x - box_w/2 - pad, y_box - pad),
                                   box_w + 2*pad, box_h + 2*pad,
                                   boxstyle="round,pad=0.05",
                                   facecolor=color, alpha=alpha, linewidth=0)
            ax.add_patch(rect)
        # box
        rect = FancyBboxPatch((x - box_w/2, y_box), box_w, box_h,
                               boxstyle="round,pad=0.05",
                               facecolor=PANEL, edgecolor=color,
                               linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x, y_box + box_h - 0.32, title_s,
                ha="center", va="top", fontsize=9,
                color=color, fontweight="bold", fontfamily=FONT)
        ax.text(x, y_box + 0.35, body,
                ha="center", va="bottom", fontsize=7.5,
                color=SEC, fontfamily=MONO, linespacing=1.5)

    # Arrows between boxes
    for i in range(len(steps) - 1):
        x1 = steps[i][0] + box_w/2 + 0.0
        x2 = steps[i+1][0] - box_w/2 - 0.0
        y_mid = y_box + box_h/2
        ax.annotate("", xy=(x2, y_mid), xytext=(x1, y_mid),
                    arrowprops=dict(arrowstyle="-|>", color=BORDER,
                                   lw=1.5, mutation_scale=14))

    # Bottom annotation: DiffMean replaced
    ax.text(6.1, 1.1,
            "← replaces DiffMean →",
            ha="center", va="center", fontsize=8, color=RED,
            fontfamily=FONT, fontstyle="italic")
    ax.annotate("", xy=(3.6, 1.3), xytext=(6.1, 1.15),
                arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.2,
                                mutation_scale=10, connectionstyle="arc3,rad=0.15"))
    ax.annotate("", xy=(8.6, 1.3), xytext=(6.1, 1.15),
                arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.2,
                                mutation_scale=10, connectionstyle="arc3,rad=-0.15"))

    # "Preserved from AlphaSteer" badge
    for x_b in [8.6, 11.1]:
        ax.text(x_b, y_box - 0.3, "✓ AlphaSteer",
                ha="center", va="top", fontsize=7, color=GREEN,
                fontfamily=FONT)

    ax.text(6.1, y_box - 0.3, "✨ AGOP direction",
            ha="center", va="top", fontsize=7, color=TEAL,
            fontfamily=FONT)

    save("fig1_method_overview.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 2 — DSR sweep: AGOPNullSpace vs AlphaSteer
# ══════════════════════════════════════════════════════════════════════════════

def fig2_dsr_sweep():
    # AlphaSteer (negative strength → flip sign for display)
    alpha_strengths = [0.1, 0.2, 0.3, 0.4, 0.5]
    alpha_data = {
        "AIM":       [100, 100, 100, 100, 100],
        "AutoDAN":   [100, 100, 100, 100, 100],
        "Cipher":    [0,   21,  41,  50,  55],
        "GCG":       [94,  94,  94,  95,  97],
        "Jailbroken":[97.0,97.0,97.4,98.8,99.8],
        "PAIR":      [86,  98, 100, 100, 100],
        "ReNeLLM":   [53,  74,  88,  98, 100],
    }

    # AGOPNullSpace (positive strength)
    agop_strengths = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85]
    agop_data = {
        "AIM":       [99, 100, 100, 100, 100, 100, 100, 100, 100, 100,  54],
        "AutoDAN":   [85, 100, 100, 100, 100, 100, 100, 100, 100,  87,  49],
        "Cipher":    [16,  14,  25,  36,  43,  51,  70,  82,  95,  98, 100],
        "GCG":       [93,  92,  93,  93,  93,  92,  92,  91,  91,  91,  93],
        "Jailbroken":[95.6,95.6,96.2,96.4,95.2,96.8,98.0,98.2,98.6,98.6,96.8],
        "PAIR":      [80,  81,  82,  88,  92, 100, 100,  99, 100,  96,  93],
        "ReNeLLM":   [47,  53,  60,  73,  77,  83,  90,  92,  97,  97,  98],
    }

    attacks = list(alpha_data.keys())
    colors  = [BLUE, TEAL, RED, PURPLE, GREEN, ORANGE, GOLD]

    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    fig.patch.set_facecolor(BG)
    fig.suptitle("DSR (%) vs Steering Strength — AGOPNullSpace vs AlphaSteer",
                 fontsize=13, color=PRI, fontweight="bold", y=0.98, fontfamily=FONT)

    for idx, (atk, color) in enumerate(zip(attacks, colors)):
        ax = axes[idx // 4][idx % 4]
        ax.set_facecolor(PANEL)
        ax.spines[:].set_color(BORDER)
        ax.tick_params(colors=SEC, labelsize=7)

        # AGOP
        ax.plot(agop_strengths, agop_data[atk], "o-",
                color=color, lw=2, ms=5, label="AGOPNullSpace")
        # AlphaSteer (shift to positive display)
        ax.plot(alpha_strengths, alpha_data[atk], "s--",
                color=SEC, lw=1.5, ms=4, alpha=0.7, label="AlphaSteer")

        ax.axhline(100, color=BORDER, lw=0.8, ls=":")
        ax.set_title(atk, fontsize=10, color=color, fontweight="bold",
                     fontfamily=FONT)
        ax.set_ylim(-2, 108)
        ax.set_xlabel("Strength ε", fontsize=7.5, color=SEC)
        ax.set_ylabel("DSR %", fontsize=7.5, color=SEC)
        ax.grid(True, color=BORDER, alpha=0.4, lw=0.6)

        # Cipher special annotation
        if atk == "Cipher":
            ax.annotate("+45pp\nat ε=0.85", xy=(0.85, 100), xytext=(0.5, 75),
                        arrowprops=dict(arrowstyle="-|>", color=GOLD, lw=1.2),
                        fontsize=7.5, color=GOLD, fontfamily=FONT)

    # Average DSR panel
    ax = axes[1][3]
    ax.set_facecolor(PANEL)
    ax.spines[:].set_color(BORDER)
    ax.tick_params(colors=SEC, labelsize=7)

    avg_agop   = [np.mean([agop_data[a][i] for a in attacks]) for i in range(len(agop_strengths))]
    avg_alpha  = [np.mean([alpha_data[a][i] for a in attacks]) for i in range(len(alpha_strengths))]
    ax.plot(agop_strengths, avg_agop,  "o-", color=BLUE, lw=2.5, ms=6, label="AGOPNullSpace")
    ax.plot(alpha_strengths, avg_alpha,"s--", color=SEC,  lw=1.8, ms=5, alpha=0.8, label="AlphaSteer")
    ax.set_title("Avg DSR (all attacks)", fontsize=10, color=BLUE,
                 fontweight="bold", fontfamily=FONT)
    ax.set_ylim(50, 105)
    ax.axhline(98.5, color=BLUE, lw=0.8, ls=":", alpha=0.5)
    ax.axhline(93.3, color=SEC,  lw=0.8, ls=":", alpha=0.5)
    ax.text(0.82, 98.5+0.5, "98.5%", color=BLUE, fontsize=7.5, fontfamily=FONT)
    ax.text(0.42, 93.3+0.5, "93.3%", color=SEC,  fontsize=7.5, fontfamily=FONT)
    ax.set_xlabel("Strength ε", fontsize=7.5, color=SEC)
    ax.set_ylabel("Avg DSR %", fontsize=7.5, color=SEC)
    ax.grid(True, color=BORDER, alpha=0.4, lw=0.6)
    ax.legend(fontsize=7, frameon=False, labelcolor=PRI)

    plt.tight_layout()
    save("fig2_dsr_sweep.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 3 — Cipher & encoding-attack highlight bar chart
# ══════════════════════════════════════════════════════════════════════════════

def fig3_cipher_highlight():
    methods = ["Baseline\n(no steering)", "Jailbreak\nAntidote", "Surgical",
               "CAST", "Circuit\nBreaker", "AlphaSteer", "AGOPNullSpace\n(Ours)"]
    cipher_dsr = [2,  0,  61, 67,  34,  55, 100]
    avg_dsr    = [48.0, 76.94, 82.83, 80.57, 84.42, 93.3, 98.5]

    x = np.arange(len(methods))
    w = 0.38

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(BG)
    fig.suptitle("Encoding-Obfuscated Attack (Cipher) DSR  &  Overall Avg DSR",
                 fontsize=12, color=PRI, fontweight="bold", fontfamily=FONT)

    bar_colors = [SEC, SEC, SEC, SEC, SEC, ORANGE, BLUE]

    for ax, vals, ylabel, title_str in [
        (ax1, cipher_dsr, "DSR % ↑", "Cipher Attack DSR %"),
        (ax2, avg_dsr,    "Avg DSR % ↑", "Average DSR (all 7 attacks)"),
    ]:
        ax.set_facecolor(PANEL)
        ax.spines[:].set_color(BORDER)
        ax.tick_params(colors=SEC, labelsize=8)
        bars = ax.bar(x, vals, color=bar_colors, width=0.6,
                      edgecolor=BG, linewidth=0.5)
        # Glow on "Ours"
        bars[-1].set_linewidth(1.5)
        bars[-1].set_edgecolor(BLUE)
        # Value labels
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.0,
                    f"{v:.0f}" if v == int(v) else f"{v:.1f}",
                    ha="center", va="bottom", fontsize=8.5,
                    color=BLUE if bar == bars[-1] else PRI,
                    fontweight="bold" if bar == bars[-1] else "normal",
                    fontfamily=FONT)
        ax.set_xticks(x)
        ax.set_xticklabels(methods, fontsize=7.8, color=SEC)
        ax.set_ylabel(ylabel, fontsize=9, color=SEC)
        ax.set_title(title_str, fontsize=10.5, color=PRI,
                     fontweight="bold", fontfamily=FONT)
        ax.set_ylim(0, 115)
        ax.axhline(100, color=BORDER, lw=0.8, ls=":")
        ax.grid(axis="y", color=BORDER, alpha=0.35, lw=0.6)
        ax.set_facecolor(PANEL)

    # Annotation on Cipher panel
    ax1.annotate("+45pp vs\nAlphaSteer",
                 xy=(6, 100), xytext=(4.5, 90),
                 arrowprops=dict(arrowstyle="-|>", color=GOLD, lw=1.3),
                 fontsize=8.5, color=GOLD, fontweight="bold", fontfamily=FONT)

    plt.tight_layout()
    save("fig3_cipher_highlight.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 4 — Radar: Safety vs Utility
# ══════════════════════════════════════════════════════════════════════════════

def fig4_radar():
    categories = ["AIM", "AutoDAN", "Cipher", "GCG", "Jailbroken",
                  "PAIR", "ReNeLLM", "XSTest CR", "MATH500", "GSM8K"]
    N = len(categories)

    # Best DSR + utility scores (normalized 0-1)
    alpha_scores = [100/100, 100/100, 55/100, 97/100, 99.8/100,
                    100/100, 100/100, 92.4/100, 45/100, 91/100]
    agop_scores  = [100/100, 100/100, 100/100, 93/100, 98.6/100,
                    100/100, 100/100, 93.6/100, 46/100, 86/100]

    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    def close(lst): return lst + lst[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.spines["polar"].set_color(BORDER)
    ax.tick_params(colors=SEC, labelsize=8.5)

    # Grid
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9, color=PRI, fontfamily=FONT)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["25", "50", "75", "100"], fontsize=7.5, color=SEC)
    ax.yaxis.set_tick_params(labelsize=7)
    for y in [0.25, 0.5, 0.75, 1.0]:
        ax.plot(angles, [y]*len(angles), color=BORDER, lw=0.7, ls=":")

    # Plot
    ax.plot(angles, close(alpha_scores), "o-", color=ORANGE, lw=2,
            ms=5, label="AlphaSteer")
    ax.fill(angles, close(alpha_scores), color=ORANGE, alpha=0.15)

    ax.plot(angles, close(agop_scores), "o-", color=BLUE, lw=2.5,
            ms=6, label="AGOPNullSpace (Ours)")
    ax.fill(angles, close(agop_scores), color=BLUE, alpha=0.2)

    ax.set_title("Safety & Utility Radar\nAGOPNullSpace vs AlphaSteer",
                 fontsize=12, color=PRI, fontweight="bold",
                 pad=25, fontfamily=FONT)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15),
              frameon=False, fontsize=9.5, labelcolor=PRI)

    save("fig4_radar.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 5 — AGOP direction vs DiffMean concept illustration
# ══════════════════════════════════════════════════════════════════════════════

def fig5_agop_concept():
    np.random.seed(0)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))
    fig.patch.set_facecolor(BG)
    fig.suptitle("Refusal Direction: DiffMean vs AGOP Top Eigenvector",
                 fontsize=12, color=PRI, fontweight="bold", fontfamily=FONT, y=1.0)

    def draw_panel(ax, title, show_agop=False):
        ax.set_facecolor(PANEL)
        ax.spines[:].set_color(BORDER)
        ax.tick_params(colors=SEC, labelsize=7)
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_xlabel("PC-1 (activation space)", fontsize=8.5, color=SEC)
        ax.set_ylabel("PC-2 (activation space)", fontsize=8.5, color=SEC)
        ax.set_title(title, fontsize=10, color=PRI, fontweight="bold",
                     fontfamily=FONT)
        ax.axhline(0, color=BORDER, lw=0.6); ax.axvline(0, color=BORDER, lw=0.6)
        ax.grid(True, color=BORDER, alpha=0.25, lw=0.5)

        # Benign cluster
        bx = np.random.randn(80)*0.8 - 1.5
        by = np.random.randn(80)*0.8
        ax.scatter(bx, by, color=GREEN, s=18, alpha=0.6, label="Benign", zorder=3)

        # Malicious cluster — standard
        mx_std = np.random.randn(80)*0.8 + 1.5
        my_std = np.random.randn(80)*0.8
        # Cipher-encoded malicious: same PC-1 but rotated
        mx_cipher = np.random.randn(40)*0.6 - 0.2
        my_cipher = np.random.randn(40)*0.6 + 2.5

        if show_agop:
            ax.scatter(np.concatenate([mx_std, mx_cipher]),
                       np.concatenate([my_std, my_cipher]),
                       color=RED, s=18, alpha=0.6, label="Malicious", zorder=3)
        else:
            ax.scatter(mx_std, my_std, color=RED, s=18, alpha=0.6,
                       label="Malicious", zorder=3)
            ax.scatter(mx_cipher, my_cipher, color=ORANGE, s=18, alpha=0.5,
                       label="Cipher-encoded", zorder=3, marker="^")

        # DiffMean direction
        mean_b = np.array([-1.5, 0.0])
        mean_m = np.array([1.5, 0.0])
        diff   = mean_m - mean_b
        diff  /= np.linalg.norm(diff)
        ax.annotate("", xy=mean_b + 2.0*diff, xytext=mean_b,
                    arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=2.5,
                                   mutation_scale=15))
        ax.text(mean_b[0] + 2.2*diff[0], mean_b[1] + 2.2*diff[1],
                "r_dim\n(DiffMean)", ha="center", fontsize=7.5,
                color=ORANGE, fontfamily=FONT)

        if show_agop:
            # AGOP eigenvec — diagonal to capture cipher cluster too
            agop_dir = np.array([0.55, 0.835])
            agop_dir /= np.linalg.norm(agop_dir)
            center = np.array([-0.1, 0.0])
            ax.annotate("", xy=center + 2.8*agop_dir, xytext=center - 1.5*agop_dir,
                        arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2.5,
                                       mutation_scale=15))
            ax.text(center[0] + 3.1*agop_dir[0], center[1] + 3.1*agop_dir[1],
                    "r_rfm\n(AGOP)", ha="center", fontsize=7.5,
                    color=BLUE, fontfamily=FONT)

            ax.text(0, -3.5,
                    "AGOP metric captures encoding-invariant boundary",
                    ha="center", fontsize=8, color=BLUE, fontstyle="italic",
                    fontfamily=FONT)
        else:
            ax.text(0, -3.5,
                    "DiffMean misses Cipher cluster (same PC-1, different PC-2)",
                    ha="center", fontsize=8, color=ORANGE, fontstyle="italic",
                    fontfamily=FONT)

        ax.legend(fontsize=7.5, frameon=False, labelcolor=PRI,
                  loc="upper left")

    draw_panel(ax1, "DiffMean (AlphaSteer)", show_agop=False)
    draw_panel(ax2, "AGOP Top Eigenvector (AGOPNullSpace)", show_agop=True)

    plt.tight_layout()
    save("fig5_agop_concept.png")


if __name__ == "__main__":
    fig1_pipeline()
    fig2_dsr_sweep()
    fig3_cipher_highlight()
    fig4_radar()
    fig5_agop_concept()
    print("\nAll figures saved to /mnt/user-data/outputs/")

/tmp/ipykernel_36356/1427288529.py:40: UserWarning: Glyph 10024 (\N{SPARKLES}) missing from font(s) DejaVu Sans.
  plt.savefig(f"/home/workspace/mad_workspace/llm/AlphaSteer/figures/{name}", dpi=dpi,


✓ fig1_method_overview.png saved
✓ fig2_dsr_sweep.png saved
✓ fig3_cipher_highlight.png saved
✓ fig4_radar.png saved
✓ fig5_agop_concept.png saved

All figures saved to /mnt/user-data/outputs/


In [18]:
"""
gen_tables.py  —  Generate Table 1 (DSR) and Table 2 (Utility) as high-quality PNGs
Run: python gen_tables.py
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

# ========== WHITE BACKGROUND THEME ==========
BG      = "#FFFFFF"      # White background
PANEL   = "#F8F9FA"      # Very light gray for panels
BORDER  = "#D1D5DB"      # Light gray border
PRI     = "#111827"      # Dark gray/black for primary text
SEC     = "#6B7280"      # Medium gray for secondary text
BLUE    = "#2563EB"      # Bright blue (high contrast)
GREEN   = "#059669"      # Emerald green
ORANGE  = "#EA580C"      # Bright orange
RED     = "#DC2626"      # Bright red
PURPLE  = "#7C3AED"      # Vibrant purple
GOLD    = "#D97706"      # Amber/gold for best values
OUR_BG  = "#EFF6FF"      # Light blue background for ours
HEAD_BG = "#E5E7EB"      # Light gray header background
FONT    = "DejaVu Sans"
MONO    = "DejaVu Sans Mono"

STYLE_COLOR = {
    "base":     PRI,
    "other":    PRI,
    "alpha":    ORANGE,
    "ablation": PURPLE,
    "ours_a":   RED,      # Attack = RED
    "ours_d":   BLUE,     # Defend = BLUE
}
STYLE_BG = {
    "base":     PANEL,
    "other":    PANEL,
    "alpha":    PANEL,
    "ablation": "#FEF3C7",      # Light amber
    "ours_a":   "#FEF2F2",      # Light red background for attack
    "ours_d":   OUR_BG,         # Light blue background for defend
}

def heat_color(val, col_vals):
    nums = [v for v in col_vals if v is not None]
    if not nums or val is None:
        return PANEL, SEC
    lo, hi = min(nums), max(nums)
    if hi == lo:
        return PANEL, PRI
    t = (val - lo) / (hi - lo)
    # Lighter colors for white background
    r = int(220 - t * 60)
    g = int(240 - t * 60)
    b = int(250 - t * 80)
    return f"#{r:02x}{g:02x}{b:02x}", PRI if t > 0.25 else SEC


def make_table(fname, title, subtitle, col_headers, col_widths, sections):
    n_data_cols = len(col_headers)
    n_rows = sum(len(rows) for _, rows in sections) + len(sections) + 1
    row_h  = 0.55          # Increased row height
    fig_w  = sum(col_widths) + 0.3
    fig_h  = n_rows * row_h + 1.8

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.axis("off")

    xs = [0.0]
    for w in col_widths[:-1]:
        xs.append(xs[-1] + w)
    total_w = sum(col_widths)

    # center x of data column ci (0-indexed among data cols)
    cx = [xs[ci + 1] + col_widths[ci + 1] / 2 for ci in range(n_data_cols)]

    def draw_rect(y, h, bg):
        r = FancyBboxPatch((0, y - h), total_w, h,
                            boxstyle="round,pad=0.01", linewidth=0.5,
                            edgecolor=BORDER, facecolor=bg, clip_on=False)
        ax.add_patch(r)

    # Collect col values for heat (exclude ablation)
    col_pool = [[] for _ in range(n_data_cols)]
    for _, rows in sections:
        for _, style, vals in rows:
            if style == "ablation":
                continue
            for ci in range(min(n_data_cols, len(vals))):
                if vals[ci] is not None:
                    col_pool[ci].append(vals[ci])

    y = (n_rows + 2.0) * row_h

    # Title - LARGER FONT
    ax.text(total_w / 2, y + 0.65, title,
            ha="center", va="bottom", fontsize=16, color=PRI,
            fontweight="bold", fontfamily=FONT)
    ax.text(total_w / 2, y + 0.25, subtitle,
            ha="center", va="bottom", fontsize=10.5, color=SEC, fontfamily=FONT)

    # Header
    draw_rect(y, row_h, HEAD_BG)
    ax.text(0.08, y - row_h / 2, "Model / Method",
            ha="left", va="center", fontsize=11, color=BLUE,
            fontweight="bold", fontfamily=FONT)
    for ci, hdr in enumerate(col_headers):
        ax.text(cx[ci], y - row_h / 2, hdr,
                ha="center", va="center", fontsize=10.5,
                color=BLUE, fontweight="bold", fontfamily=FONT)
    y -= row_h

    for sec_label, rows in sections:
        draw_rect(y, row_h, "#F3F4F6")  # Slightly darker than PANEL
        ax.text(0.08, y - row_h / 2, sec_label,
                ha="left", va="center", fontsize=10.5, color=PRI,
                fontweight="bold", fontstyle="italic", fontfamily=FONT)
        y -= row_h

        for row_label, style, vals in rows:
            draw_rect(y, row_h, STYLE_BG[style])
            
            # Đặc biệt cho AGOPNullSpace: thêm khung nổi bật
            text_color = STYLE_COLOR[style]
            if style in ("ours_a", "ours_d"):
                # Tăng font weight và kích thước cho AGOPNullSpace
                fontweight = "bold"
                fontsize_label = 10.5  # Lớn hơn một chút
                # Thêm viền đậm hơn (sẽ add sau)
                border_patch = FancyBboxPatch((0, y - row_h), total_w, row_h,
                                              boxstyle="round,pad=0.01", linewidth=2,
                                              edgecolor=text_color, facecolor="none", clip_on=False)
                ax.add_patch(border_patch)
            else:
                fontweight = "bold" if style in ("ablation") else "normal"
                fontsize_label = 10
            
            ax.text(0.10, y - row_h / 2, row_label,
                    ha="left", va="center", fontsize=fontsize_label,
                    color=text_color,
                    fontweight=fontweight,
                    fontfamily=FONT)

            for ci in range(n_data_cols):
                v = vals[ci] if ci < len(vals) else None
                if v is None:
                    ax.text(cx[ci], y - row_h / 2, "—",
                            ha="center", va="center", fontsize=11,
                            color=SEC, fontfamily=MONO)
                    continue
                txt = f"{v:.0f}" if isinstance(v, float) and v == int(v) else \
                      f"{v:.1f}" if isinstance(v, float) else str(v)
                
                # CHO AGOPNullSpace: ưu tiên màu của style (đỏ cho attack, xanh cho defend)
                if style in ("ours_a", "ours_d"):
                    # Highlight màu theo attack/defend, bỏ qua heat color
                    clr = text_color
                    # Thêm đậm hơn
                    fontweight_cell = "bold"
                    fontsize_cell = 12  # To hơn các số khác
                else:
                    # Các dòng khác dùng heat color bình thường
                    _, clr = heat_color(v, col_pool[ci])
                    if col_pool[ci] and v == max(col_pool[ci]) and style != "ablation":
                        clr = GOLD
                    fontweight_cell = "bold" if style in ("ours_d","ours_a") else "normal"
                    fontsize_cell = 11
                
                ax.text(cx[ci], y - row_h / 2, txt,
                        ha="center", va="center", fontsize=fontsize_cell,
                        color=clr,
                        fontweight=fontweight_cell,
                        fontfamily=MONO)
            y -= row_h

        ax.axhline(y + row_h * 0.08, color=BORDER, linewidth=0.8)

    ax.set_xlim(0, total_w)
    ax.set_ylim(y - 0.5, (n_rows + 3.8) * row_h)

    legend_patches = [
        mpatches.Patch(color=BLUE,   label="Ours (AGOPNullSpace) - Defend"),
        mpatches.Patch(color=RED,    label="Ours (AGOPNullSpace) - Attack"),
        mpatches.Patch(color=ORANGE, label="AlphaSteer"),
        mpatches.Patch(color=PURPLE, label="RV Ablation"),
        mpatches.Patch(color=GOLD,   label="Best in column"),
        mpatches.Patch(color=GREEN,  label="High score"),
    ]
    ax.legend(handles=legend_patches, loc="lower center", ncol=7,
              frameon=False, fontsize=9, labelcolor=PRI,
              bbox_to_anchor=(0.5, -0.08))

    plt.tight_layout(pad=0.2)
    plt.savefig(fname, dpi=200, bbox_inches="tight", facecolor=BG)
    print(f"  {fname} saved")
    plt.close()


def make_table1():
    attacks = ["AIM", "AutoDAN", "Cipher", "GCG", "Jailbroken", "PAIR", "ReNeLLM", "Avg DSR"]
    col_w   = [3.9, 0.95, 1.05, 0.95, 0.9, 1.15, 0.9, 1.1, 1.0]

    sections = [
        ("  Judge: GPT-4o", [
            ("Llama-3.1-8B-Instruct",          "base",    [92.0, 48.0, 0.0,  58.0, 75.0, 45.0,  28.0, 48.00]),
            ("+ Jailbreak Antidote",            "other",   [100.0,97.0, 0.0, 100.0, 86.0, 93.0,  63.0, 76.94]),
            ("+ Surgical",                      "other",   [100.0,76.0,61.0,  98.0, 88.0, 90.0,  67.0, 82.83]),
            ("+ CAST",                          "other",   [92.0, 51.0,67.0,  99.0, 81.0, 96.0,  96.0, 80.57]),
            ("+ Circuit Breaker",               "other",   [100.0,100.0,34.0,100.0, 80.0, 96.0,  81.0, 84.42]),
            ("+ RV Ablation",                   "ablation",[100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.00]),
            ("+ AlphaSteer",                    "alpha",   [100.0, 99.0,63.0, 97.0, 92.0, 98.0, 100.0, 91.93]),
        ]),
        ("  Judge: Llama-Guard-4-12B", [
            ("Llama-3.1-8B-Instruct",           "base",    [93.0, 51.0,  2.0, 63.0, 92.4, 72.0,  45.0, 59.77]),
            ("+ AlphaSteer",                    "alpha",   [100.0,100.0,55.0, 97.0, 99.8,100.0, 100.0, 93.11]),
            ("+ AGOPNullSpace Attack  (Ours)",  "ours_a",  [9.0,  24.0,  9.0, 86.0, 78.6, 71.0,  23.0, 42.94]),
            ("+ AGOPNullSpace Defend  (Ours)",  "ours_d",  [100.0,100.0,100.0,93.0, 98.0,100.0, 100.0, 98.71]),
        ]),
    ]
    make_table("/home/workspace/mad_workspace/llm/AlphaSteer/figures/table1_dsr.png",
               "Table 1: Jailbreak Attack DSR ↑ — Llama-3.1-8B-Instruct",
               "Defense Success Rate % (higher = better)  •  Bold = AGOPNullSpace  •  Gold = best per column",
               attacks, col_w, sections)


def make_table2():
    metrics = ["XSTest\nCR%↑", "AlpacaEval\nWR%↑", "MATH500\nAcc%↑", "GSM8K\nAcc%↑", "Utility\nScore%"]
    col_w   = [3.6, 1.35, 1.5, 1.4, 1.35, 1.4]

    sections = [
        ("  Judge: GPT-4o", [
            ("Llama-3.1-8B-Instruct",         "base",    [92.4, 50.0, 45.0, 81.0, 67.1]),
            ("+ Jailbreak Antidote",           "other",   [84.8, 47.3, 43.0, 81.0, 64.0]),
            ("+ Surgical",                     "other",   [62.0, 47.0, 48.0, 80.0, 59.3]),
            ("+ CAST",                         "other",   [90.0, 31.1,  0.0,  0.0, 30.2]),
            ("+ Circuit Breaker",              "other",   [84.8, 23.7, 18.0, 48.0, 43.6]),
            ("+ RV Ablation",                  "ablation",[4.0,  10.4, 37.0, 65.0, 29.1]),
            ("+ AlphaSteer",                   "alpha",   [91.2, 48.1, 46.0, 84.0, 67.3]),
        ]),
        ("  Judge: Llama-Guard-4-12B", [
            ("Llama-3.1-8B-Instruct",         "base",    [85.0, None, 45.0, 93.2, None]),
            ("+ AlphaSteer",                   "alpha",   [91.0, None, 45.0, 92.4, None]),
            ("+ AGOPNullSpace Attack (Ours)",  "ours_a",  [85.0, None, 48.0, 93.6, None]),
            ("+ AGOPNullSpace Defend (Ours)",  "ours_d",  [87.0, None, 46.0, 93.6, None]),
        ]),
    ]
    make_table("/home/workspace/mad_workspace/llm/AlphaSteer/figures/table2_utility.png",
               "Table 2: Performance on Utility Benchmarks — Llama-3.1-8B-Instruct",
               "↑ higher is better  •  — = not evaluated  •  AGOPNullSpace preserves or improves utility",
               metrics, col_w, sections)


if __name__ == "__main__":
    make_table1()
    make_table2()
    print("Done.")

  /home/workspace/mad_workspace/llm/AlphaSteer/figures/table1_dsr.png saved
  /home/workspace/mad_workspace/llm/AlphaSteer/figures/table2_utility.png saved
Done.


In [ ]:
#!/bin/bash

FILES=(
    "./data/responses/llama3.1_GPT4-o-Judge/aim_llama3.1_rfm_results.json"
    "./data/responses/llama3.1_GPT4-o-Judge/autodan_llama3.1_rfm_results.json"
    "./data/responses/llama3.1_GPT4-o-Judge/cipher_llama3.1_rfm_results.json"
    "./data/responses/llama3.1_GPT4-o-Judge/gcg_llama3.1_rfm_results.json"
    "./data/responses/llama3.1_GPT4-o-Judge/jailbroken_llama3.1_rfm_results.json"
    "./data/responses/llama3.1_GPT4-o-Judge/pair_llama3.1_rfm_results.json"
    "./data/responses/llama3.1_GPT4-o-Judge/renellm_llama3.1_rfm_results.json"
)

for FILE in "${FILES[@]}"; do
    echo "Processing: $FILE"
    python eval.py --input-file "$FILE" --model openai/gpt-4o
    echo "Done: $FILE"
    echo "---"
done
